In [0]:
%run ./Includes/Copy-Datasets

In [0]:
%sql
SELECT CAST(key AS STRING), CAST(value AS STRING) FROM bronze LIMIT 20;

In [0]:
%sql
SELECT v.*
FROM (
  SELECT from_json(CAST(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v 
  FROM bronze
  WHERE topic = "orders"
)

In [0]:
(spark.readStream
        .table("bronze")
        .createOrReplaceTempView("bronze_tmp"))

In [0]:
%sql
SELECT v.*
FROM (
  SELECT from_json(CAST(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v 
  FROM bronze_tmp
  WHERE topic = "orders"
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_silver_tmp AS
SELECT v.*
FROM (
  SELECT from_json(CAST(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v 
  FROM bronze_tmp
  WHERE topic = "orders"
)

In [0]:
query = (spark.table("orders_silver_tmp")
                .writeStream
                .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/orders_silver")
                .trigger(availableNow=True)
                .table("orders_silver")        
        )
query.awaitTermination()

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream
                .table("bronze")
                .filter("topic='orders'")
                .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
                .select("v.*")
            .writeStream
                .option("checkpointLocation", "dbfs:/mnt/demo_pro/checkpoints/orders_silver")
                .trigger(availableNow=True)
                .table("orders_silver")
        )
query.awaitTermination()

In [0]:
%sql
SELECT * FROM orders_silver